In [44]:
import numpy as np
from PIL import Image
from numba import cuda
import time

print("CUDA disponible :", cuda.is_available())

CUDA disponible : True


In [45]:
@cuda.jit  #pour transforme la fonction en kernel destiné au GPU 
def coordinates2D():   #pour  Créer le kernel coordinates2D()  
    tx = cuda.threadIdx.x   #  la  position horizontale du thread dans son bloc. 
    ty = cuda.threadIdx.y   # verticale 
 
    print(tx, ty)  # Affichage du deux position 

In [46]:
coordinates2D[(1, 1), (4, 4)]()  #Exécuter avec un bloc de [ 4 × 4 ]threads
cuda.synchronize()  

c:\Users\GiGi\Desktop\TPAML\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [47]:
@cuda.jit
def coordinatesGlobal2D():

    # Position locale
    tx = cuda.threadIdx.x
    ty = cuda.threadIdx.y

    # Position globale automatique ////  # calcule automatiquement 
    # les coordonnées globales
    x, y = cuda.grid(2)

    # Position globale manuelle
    x_manual = (
        cuda.blockIdx.x * cuda.blockDim.x
        + cuda.threadIdx.x
    )

    y_manual = (
        cuda.blockIdx.y * cuda.blockDim.y
        + cuda.threadIdx.y
    )

    print(tx, ty, x, y, x_manual, y_manual)

In [48]:
coordinatesGlobal2D[(1, 1), (4, 4)]()
cuda.synchronize()

c:\Users\GiGi\Desktop\TPAML\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))


In [49]:
# Créer une matrice de 16 × 16
H = 16
W = 16

matrix = np.zeros((H, W), dtype=np.int32) #une matrice remplie par des 0 

d_matrix = cuda.to_device(matrix) #copie la matrice de la RAM vers la mémoire du GPU  

In [50]:
@cuda.jit
def fillMatrix(matrix):

    x, y = cuda.grid(2)

    H, W = matrix.shape

    if x < W and y < H:
        matrix[y, x] = y * W + x

In [ ]:
threads_per_block = (8, 8)  # la taille du block 

blocks_x = (W + 7) // 8
blocks_y = (H + 7) // 8

grid_dim = (blocks_x, blocks_y)  # nbr du bloc

fillMatrix[grid_dim, threads_per_block](d_matrix)

cuda.synchronize()

result = d_matrix.copy_to_host()

print(result)

[[  0   1   2   3   4   5   6   7   8   9  10  11  12  13  14  15]
 [ 16  17  18  19  20  21  22  23  24  25  26  27  28  29  30  31]
 [ 32  33  34  35  36  37  38  39  40  41  42  43  44  45  46  47]
 [ 48  49  50  51  52  53  54  55  56  57  58  59  60  61  62  63]
 [ 64  65  66  67  68  69  70  71  72  73  74  75  76  77  78  79]
 [ 80  81  82  83  84  85  86  87  88  89  90  91  92  93  94  95]
 [ 96  97  98  99 100 101 102 103 104 105 106 107 108 109 110 111]
 [112 113 114 115 116 117 118 119 120 121 122 123 124 125 126 127]
 [128 129 130 131 132 133 134 135 136 137 138 139 140 141 142 143]
 [144 145 146 147 148 149 150 151 152 153 154 155 156 157 158 159]
 [160 161 162 163 164 165 166 167 168 169 170 171 172 173 174 175]
 [176 177 178 179 180 181 182 183 184 185 186 187 188 189 190 191]
 [192 193 194 195 196 197 198 199 200 201 202 203 204 205 206 207]
 [208 209 210 211 212 213 214 215 216 217 218 219 220 221 222 223]
 [224 225 226 227 228 229 230 231 232 233 234 235 236 237 238 

c:\Users\GiGi\Desktop\TPAML\.venv\Lib\site-packages\numba_cuda\numba\cuda\dispatcher.py:748: NumbaPerformanceWarning: Grid size 4 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
